# 02: Metrics Monitoring

This notebook covers tracking key AI metrics: latency, token usage, and cost. You'll build monitoring wrappers and learn to identify bottlenecks.

## Objectives
- Track latency, token usage, and cost programmatically
- Build a monitoring wrapper for LLM API calls
- Estimate cost per request
- Identify slow operations
- Create a metrics summary

## 1. Setting Up the Monitoring Infrastructure

In [ ]:
import os
import time
import json
from dataclasses import dataclass, field, asdict
from typing import Optional, Any
from datetime import datetime, timedelta
from collections import defaultdict
import statistics

# Set up API key
os.environ["OPENAI_API_KEY"] = "your-openai-api-key"  # Replace with your key

## 2. Building a Cost Tracking Data Model

In [ ]:
@dataclass
class MetricRecord:
    """A single metric record for an LLM call."""
    timestamp: float
    model: str
    input_tokens: int
    output_tokens: int
    latency_ms: float
    cost_usd: float
    operation: str = "llm_call"
    metadata: dict = field(default_factory=dict)
    
    @property
    def total_tokens(self) -> int:
        return self.input_tokens + self.output_tokens
    
    @property
    def datetime(self) -> datetime:
        return datetime.fromtimestamp(self.timestamp)

# Model pricing (USD per 1M tokens)
MODEL_PRICING = {
    "gpt-4": {"input": 30.0, "output": 60.0},
    "gpt-4-turbo": {"input": 10.0, "output": 30.0},
    "gpt-3.5-turbo": {"input": 0.5, "output": 1.5},
    "gpt-4o": {"input": 2.5, "output": 10.0},
    "gpt-4o-mini": {"input": 0.15, "output": 0.6},
}

def calculate_cost(model: str, input_tokens: int, output_tokens: int) -> float:
    """Calculate cost in USD for a given model and token usage."""
    pricing = MODEL_PRICING.get(model, MODEL_PRICING["gpt-4"])
    cost = (input_tokens * pricing["input"] + 
            output_tokens * pricing["output"]) / 1_000_000
    return round(cost, 6)

## 3. Building a Monitoring Wrapper

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage

class MonitoredLLM:
    """LLM wrapper with built-in metrics collection."""
    
    def __init__(self, model: str = "gpt-4o-mini", **kwargs):
        self.llm = ChatOpenAI(model=model, **kwargs)
        self.model = model
        self.metrics: list[MetricRecord] = []
    
    def invoke(self, messages: list[BaseMessage], **kwargs) -> dict:
        """Invoke LLM with metrics tracking."""
        start_time = time.time()
        
        try:
            response = self.llm.invoke(messages, **kwargs)
            latency_ms = (time.time() - start_time) * 1000
            
            # Extract token usage from response metadata
            usage = response.response_metadata.get("token_usage", {})
            input_tokens = usage.get("prompt_tokens", 0)
            output_tokens = usage.get("completion_tokens", 0)
            
            cost = calculate_cost(self.model, input_tokens, output_tokens)
            
            record = MetricRecord(
                timestamp=time.time(),
                model=self.model,
                input_tokens=input_tokens,
                output_tokens=output_tokens,
                latency_ms=latency_ms,
                cost_usd=cost,
                metadata={"status": "success"}
            )
            
        except Exception as e:
            latency_ms = (time.time() - start_time) * 1000
            record = MetricRecord(
                timestamp=time.time(),
                model=self.model,
                input_tokens=0,
                output_tokens=0,
                latency_ms=latency_ms,
                cost_usd=0,
                metadata={"status": "error", "error": str(e)}
            )
            raise
        finally:
            self.metrics.append(record)
        
        return {
            "response": response,
            "metrics": record
        }
    
    def get_metrics_summary(self) -> dict:
        """Get summary of all collected metrics."""
        if not self.metrics:
            return {"error": "No metrics collected"}
        
        latencies = [m.latency_ms for m in self.metrics]
        costs = [m.cost_usd for m in self.metrics]
        tokens = [m.total_tokens for m in self.metrics]
        
        return {
            "total_requests": len(self.metrics),
            "total_cost_usd": sum(costs),
            "avg_latency_ms": statistics.mean(latencies),
            "p95_latency_ms": sorted(latencies)[int(len(latencies) * 0.95)] if len(latencies) > 1 else latencies[0],
            "total_tokens": sum(tokens),
            "avg_tokens_per_request": statistics.mean(tokens),
            "models_used": list(set(m.model for m in self.metrics))
        }

## 4. Using the Monitoring Wrapper

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

# Create monitored LLM
monitored = MonitoredLLM(model="gpt-4o-mini", temperature=0.7)

# Run several requests
requests = [
    "What is machine learning?",
    "Explain quantum computing in one paragraph.",
    "What are the SOLID principles in software engineering?",
    "How does gradient descent work?",
    "What is a neural network?"
]

results = []
for req in requests:
    result = monitored.invoke([
        SystemMessage(content="You are a helpful technical assistant."),
        HumanMessage(content=req)
    ])
    metrics = result["metrics"]
    print(f"Q: {req[:50]}...")
    print(f"   Tokens: {metrics.input_tokens}in/{metrics.output_tokens}out | "
          f"Latency: {metrics.latency_ms:.0f}ms | "
          f"Cost: ${metrics.cost_usd:.4f}\n")
    results.append(result)

## 5. Analyzing Metrics

In [ ]:
# Get summary
summary = monitored.get_metrics_summary()
print("=== Metrics Summary ===")
for key, value in summary.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

In [ ]:
# Detailed latency analysis
print("=== Latency Analysis ===")
latencies = [m.latency_ms for m in monitored.metrics]
print(f"Min: {min(latencies):.0f}ms")
print(f"Max: {max(latencies):.0f}ms")
print(f"Mean: {statistics.mean(latencies):.0f}ms")
print(f"Median: {statistics.median(latencies):.0f}ms")
if len(latencies) > 1:
    print(f"Stdev: {statistics.stdev(latencies):.0f}ms")

# Find slowest request
slowest = max(monitored.metrics, key=lambda m: m.latency_ms)
print(f"\nSlowest request: {slowest.latency_ms:.0f}ms")
print(f"  Model: {slowest.model}")
print(f"  Tokens: {slowest.total_tokens}")

In [ ]:
# Cost breakdown
print("=== Cost Analysis ===")
total_cost = sum(m.cost_usd for m in monitored.metrics)
print(f"Total cost: ${total_cost:.4f}")
print(f"Average per request: ${total_cost/len(monitored.metrics):.4f}")
print(f"Projected daily (1000 req): ${total_cost/len(monitored.metrics) * 1000:.2f}")
print(f"Projected monthly (30k req): ${total_cost/len(monitored.metrics) * 30000:.2f}")

## 6. Cost Estimation Calculator

In [ ]:
class CostEstimator:
    """Estimate costs for different scenarios."""
    
    def __init__(self):
        self.pricing = MODEL_PRICING
    
    def estimate_single_request(self, model: str, input_tokens: int, 
                                 output_tokens: int) -> dict:
        """Estimate cost for a single request."""
        cost = calculate_cost(model, input_tokens, output_tokens)
        return {
            "model": model,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "cost_usd": cost
        }
    
    def estimate_monthly(self, model: str, avg_input_tokens: int,
                         avg_output_tokens: int, requests_per_day: int) -> dict:
        """Estimate monthly cost."""
        cost_per_request = calculate_cost(model, avg_input_tokens, avg_output_tokens)
        daily_cost = cost_per_request * requests_per_day
        monthly_cost = daily_cost * 30
        
        return {
            "model": model,
            "cost_per_request": cost_per_request,
            "daily_cost": daily_cost,
            "monthly_cost": monthly_cost,
            "requests_per_day": requests_per_day
        }
    
    def compare_models(self, input_tokens: int, output_tokens: int) -> list[dict]:
        """Compare costs across models."""
        comparisons = []
        for model in self.pricing:
            cost = calculate_cost(model, input_tokens, output_tokens)
            comparisons.append({
                "model": model,
                "cost_usd": cost,
                "cost_per_1k_requests": cost * 1000
            })
        return sorted(comparisons, key=lambda x: x["cost_usd"])

# Example usage
estimator = CostEstimator()

# Compare models for a typical request
print("=== Model Cost Comparison (500 input / 200 output tokens) ===")
comparisons = estimator.compare_models(500, 200)
for c in comparisons:
    print(f"{c['model']:20} ${c['cost_usd']:.4f} per request | "
          f"${c['cost_per_1k_requests']:.2f} per 1K requests")

# Monthly estimate
print("\n=== Monthly Estimate (GPT-4o, 500 req/day) ===")
monthly = estimator.estimate_monthly("gpt-4o", 500, 200, 500)
print(f"Cost per request: ${monthly['cost_per_request']:.4f}")
print(f"Daily cost: ${monthly['daily_cost']:.2f}")
print(f"Monthly cost: ${monthly['monthly_cost']:.2f}")

## 7. Anomaly Detection

In [ ]:
class AnomalyDetector:
    """Detect anomalies in cost and latency patterns."""
    
    def __init__(self, window_size: int = 20, threshold: float = 2.0):
        self.window_size = window_size
        self.threshold = threshold
        self.cost_history: list[float] = []
        self.latency_history: list[float] = []
    
    def check_cost(self, cost: float) -> dict:
        """Check if cost is anomalous."""
        is_anomaly = False
        mean = stdev = 0
        
        if len(self.cost_history) >= self.window_size:
            mean = statistics.mean(self.cost_history)
            stdev = statistics.stdev(self.cost_history) if len(self.cost_history) > 1 else 0
            if stdev > 0:
                z_score = (cost - mean) / stdev
                is_anomaly = abs(z_score) > self.threshold
        
        self.cost_history.append(cost)
        if len(self.cost_history) > self.window_size:
            self.cost_history.pop(0)
        
        return {
            "is_anomaly": is_anomaly,
            "cost": cost,
            "mean": mean,
            "stdev": stdev
        }
    
    def check_latency(self, latency: float) -> dict:
        """Check if latency is anomalous."""
        is_anomaly = False
        mean = stdev = 0
        
        if len(self.latency_history) >= self.window_size:
            mean = statistics.mean(self.latency_history)
            stdev = statistics.stdev(self.latency_history) if len(self.latency_history) > 1 else 0
            if stdev > 0:
                z_score = (latency - mean) / stdev
                is_anomaly = abs(z_score) > self.threshold
        
        self.latency_history.append(latency)
        if len(self.latency_history) > self.window_size:
            self.latency_history.pop(0)
        
        return {
            "is_anomaly": is_anomaly,
            "latency": latency,
            "mean": mean,
            "stdev": stdev
        }

# Test anomaly detection
detector = AnomalyDetector(window_size=5, threshold=1.5)

# Simulate normal requests
print("=== Anomaly Detection Test ===")
test_latencies = [200, 210, 195, 205, 200,  # normal
                   800,  # anomaly!
                   210, 205,  # back to normal
                   1500]  # another anomaly!

for lat in test_latencies:
    result = detector.check_latency(lat)
    status = "⚠️ ANOMALY" if result["is_anomaly"] else "✓ Normal"
    print(f"Latency: {lat:5}ms | {status} | Mean: {result['mean']:.0f}ms")

## 8. Practical Exercise

### Task
1. Extend the `MonitoredLLM` class to:
   - Track which prompts are used
   - Calculate cost savings by model (e.g., GPT-4 vs GPT-4o-mini)
   - Add rate limiting awareness

2. Build a monitoring dashboard function that:
   - Shows costs over time
   - Identifies the most expensive operations
   - Suggests optimization opportunities

In [ ]:
# Your code here
# Extend MonitoredLLM or create a new monitoring class

## Summary

In this notebook, you learned:

1. **Cost Tracking** — Calculating costs based on model pricing
2. **Monitoring Wrapper** — Building reusable instrumentation
3. **Latency Analysis** — Measuring and analyzing response times
4. **Cost Estimation** — Projecting costs for different scenarios
5. **Anomaly Detection** — Identifying unusual patterns

### Key Takeaways
- Always track token usage for cost management
- Use p95 latency for SLA monitoring
- Model selection significantly impacts cost
- Anomaly detection helps catch issues early

### Next Steps
- Proceed to Notebook 03 for evaluation metrics
- Integrate monitoring into your production systems